# Trend-following systems

European, American, and generalized TSMOM implementations from Sepp and Lucic, *The Science and Practice of Trend-Following Systems* (SSRN 3167787). The strategy converts intraday FX mid quotes to daily bars ending at 17:00 Eastern, then holds each daily target weight on the original 15-minute quote grid.

`AggressiveTrader` preserves the repository's existing backtest pattern but accrues log-return PnL, while the paper states its equations in simple returns. The strategy holds the American entry notional unchanged until exit, matching the authors' reference implementation and this repository's trader abstraction. Same-close position assignment makes the close-$t$ weight earn the next quote return, as in the paper; the trader's existing 17:00 TN carry convention applies carry to the current timestamp's position.

In [ ]:
import sys
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from data.daily_data_loader import get_fwdpts
from data.resampled_data_loader import get_resampled_quotes

start_date = dt.date(2010, 1, 1)
end_date = dt.date(2025, 12, 31)

traded_instruments = [
    "AUDUSD",
    "EURUSD",
    "GBPUSD",
    "EURGBP",
    "EURJPY",
    "USDJPY",
    "USDCHF",
    "EURCHF",
    "USDCAD",
    "USDNOK",
    "USDSEK",
    "EURNOK",
    "EURSEK",
    "NZDUSD",
]

freq = pd.Timedelta(minutes=15)
fx_price_dict = get_resampled_quotes(
    traded_instruments, start_date, end_date, freq
)
fwdpts_tn_dict = get_fwdpts(
    traded_instruments, "tn", start_date, end_date
)

In [ ]:
# Reload while iterating on the strategy.
import importlib

import strategy.tf_strategy as tfs
import trading.aggressive_trader as at

importlib.reload(tfs)
importlib.reload(at)

In [ ]:
common_params = {
    "trade_entry_time": "17:00",
    "vol_span_days": 33,
    "vol_target": 0.15,
    "annualization_factor": 260,
    "warmup_days": 250,
}

strategies = {
    "European": tfs.TFStrategy(
        traded_instruments,
        fx_price_dict,
        {
            **common_params,
            "tf_style": "European",
            "long_span_days": 250,
            "short_span_days": 20,
        },
    ),
    "American": tfs.TFStrategy(
        traded_instruments,
        fx_price_dict,
        {
            **common_params,
            "tf_style": "American",
            "long_span_days": 250,
            "short_span_days": 20,
            "atr_window_days": 33,
            "entry_atr_multiplier": 5.0,
            "stop_atr_multiplier": 5.0,
            # R is not fixed in the paper; 1% matches the authors' reference code.
            "risk_multiplier": 0.01,
        },
    ),
    "TSMOM": tfs.TFStrategy(
        traded_instruments,
        fx_price_dict,
        {
            **common_params,
            "tf_style": "TSMOM",
            "period_length_days": 10,
            "num_periods": 10,
        },
    ),
}

In [ ]:
signals_by_system = {
    system_name: strategy.generate_signals()
    for system_name, strategy in strategies.items()
}

In [ ]:
from analytics.performance_report import PerformanceReport

traders = {
    system_name: at.AggressiveTrader(
        traded_instruments,
        fx_price_dict,
        signals,
        execute_on_next_price_tick=False,
        fwdpts_tn_dict=fwdpts_tn_dict,
    )
    for system_name, signals in signals_by_system.items()
}

reports = {
    system_name: PerformanceReport(
        traded_instruments, trader
    ).generate_performance_report(include_portfolio=True)
    for system_name, trader in traders.items()
}
pd.concat(reports, names=["System", "Instrument"])

In [ ]:
freq_mult = pd.Timedelta(days=1) / freq
fig, axes = plt.subplots(len(traders), 1, figsize=(12, 12), sharex=True)

for ax, (system_name, trader) in zip(axes, traders.items()):
    net_df = pd.DataFrame(trader.generate_net_pnl()).fillna(0.0)
    gross_df = pd.DataFrame(trader.generate_gross_pnl()).fillna(0.0)
    avg_net = net_df.mean(axis=1)
    avg_gross = gross_df.mean(axis=1)

    net_std = avg_net.std()
    gross_std = avg_gross.std()
    net_sharpe = (
        np.sqrt(252 * freq_mult) * avg_net.mean() / net_std
        if net_std != 0
        else np.nan
    )
    gross_sharpe = (
        np.sqrt(252 * freq_mult) * avg_gross.mean() / gross_std
        if gross_std != 0
        else np.nan
    )

    avg_net.cumsum().plot(
        ax=ax, label=f"Net (Sharpe={net_sharpe:.2f})"
    )
    avg_gross.cumsum().plot(
        ax=ax, label=f"Gross (Sharpe={gross_sharpe:.2f})"
    )
    ax.set_title(f"{system_name}: average cumulative PnL")
    ax.set_ylabel("Cumulative PnL")
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time")
fig.tight_layout()

In [ ]:
from analytics.plot_daily_recap import plot_daily_recap

system_name = "European"
instrument = "GBPUSD"
date_range = (dt.date(2025, 1, 15), dt.date(2025, 10, 3))
plot_daily_recap(
    instrument,
    date_range,
    fx_price_dict[instrument],
    strategies[system_name],
    traders[system_name],
)